In [2]:
"""
Implementation of Algorithm 1 from:
"A Rigorous Quantum Communication Framework for Optical Fibre Channels"
by Kaumud Sharma

Algorithm 1: Adaptive Bayesian Channel Parameter Estimation

Bugs fixed
----------
B1  likelihood() returned 1e-300 for n_bar <= 0 instead of 0.0,
    causing invalid particles to retain non-zero weight.
B2  Convergence table step labels were off-by-one (1,11,21... instead of 10,20,30...).
B5  Clarified that Bayesian posterior < frequentist QCRB is correct behaviour
    because the prior supplies additional information.
"""

import math
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
from scipy.stats import multivariate_normal
import matplotlib.pyplot as plt


# =============================================================================
# CHANNEL MODEL
# =============================================================================

class FibreChannel:
    """
    Combined amplitude-damping + phase-damping optical fibre channel.
    Noise parameters from Definition III.1 (Eq. 15):
      eta      = exp(-gamma_loss * L)    -- transmission efficiency
      exp_deph = exp(-2 * gamma_phi * L) -- dephasing envelope
    Parameters estimated: Theta = [n_bar, phi].
    """

    def __init__(self, gamma_loss: float, gamma_phi: float, L: float):
        self.gamma_loss = gamma_loss
        self.gamma_phi  = gamma_phi
        self.L          = L
        self._eta       = math.exp(-gamma_loss * L)

    def qfim(self, n_bar: float) -> np.ndarray:
        """QFIM for Theta = (n_bar, phi) -- Theorem V.5, Eq. (40)."""
        eta  = self._eta
        F_nn = (1.0 / (eta * n_bar)
                + 4.0 * self.gamma_phi**2 * self.L**2 / (eta * n_bar))
        F_pp = 4.0 * eta * n_bar * math.exp(-2.0 * self.gamma_phi * self.L)
        return np.array([[F_nn, 0.0],
                         [0.0,  F_pp]])

    def sample_outcome(self, theta_true: np.ndarray,
                       rng: np.random.Generator) -> np.ndarray:
        """Draw one QCRB-saturating outcome (optimal POVM, Theorem V.6)."""
        cov = np.linalg.inv(self.qfim(theta_true[0]))
        return rng.multivariate_normal(theta_true, cov)

    def likelihood(self, outcome: np.ndarray, theta_i: np.ndarray) -> float:
        """
        p(outcome | theta_i).
        FIX B1: returns 0.0 for n_bar <= 0 (was 1e-300, biasing the posterior).
        """
        if theta_i[0] <= 0.0:
            return 0.0                          # B1 fixed
        cov = np.linalg.inv(self.qfim(theta_i[0]))
        return float(multivariate_normal.pdf(outcome, mean=theta_i, cov=cov))


# =============================================================================
# PRIOR
# =============================================================================

class GaussianPrior:
    """Gaussian prior p_0(Theta) over Theta = [n_bar, phi]."""

    def __init__(self, mean: np.ndarray, cov: np.ndarray):
        self.mean = np.asarray(mean, dtype=float)
        self.cov  = np.asarray(cov,  dtype=float)

    def sample(self, n: int, rng: np.random.Generator) -> np.ndarray:
        return rng.multivariate_normal(self.mean, self.cov, size=n)


# =============================================================================
# RESULT CONTAINER
# =============================================================================

@dataclass
class EstimationResult:
    theta_hat   : np.ndarray
    cov_final   : np.ndarray
    history_mean: list = field(default_factory=list)
    history_cov : list = field(default_factory=list)


# =============================================================================
# ALGORITHM 1
# =============================================================================

def _weighted_stats(particles: np.ndarray,
                    weights: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    mean = np.average(particles, weights=weights, axis=0)
    diff = particles - mean
    cov  = np.einsum('i,ij,ik->jk', weights, diff, diff)
    return mean, cov


def algorithm1_adaptive_bayesian_estimation(
    channel    : FibreChannel,
    prior      : GaussianPrior,
    M          : int,
    theta_true : np.ndarray,
    n_particles: int = 300,
    seed       : Optional[int] = 42,
) -> EstimationResult:
    """
    Algorithm 1 — Adaptive Bayesian Channel Parameter Estimation.

    Pseudocode (paper, Section V-C):
    ─────────────────────────────────
    Initialise k <- 0, p_k(Theta) <- p_0(Theta)
    while k < M:
        F_bar  <- integral F(Theta) p_k(Theta) dTheta
        Choose POVM maximising Tr[F_bar^{-1} F_Pi(Theta)]
        Apply POVM to copy k+1; record outcome y_{k+1}
        p_{k+1}(Theta) propto p_k(Theta) Tr[Pi_{y_{k+1}} rho_Theta]
        k <- k + 1
    Theta_hat <- integral Theta p_M(Theta) dTheta
    return Theta_hat, Cov[p_M]

    Note (B5): Bayesian posterior may fall below the frequentist QCRB
    (1/M) F^{-1}(theta_true) because the prior provides extra information.
    """
    rng = np.random.default_rng(seed)

    particles = prior.sample(n_particles, rng)
    weights   = np.ones(n_particles) / n_particles
    history_mean, history_cov = [], []

    for _ in range(M):
        # Expected QFIM: F_bar = sum_i w_i * F(Theta_i)
        F_bar = np.zeros((2, 2))
        for w_i, th_i in zip(weights, particles):
            if th_i[0] > 0.0:
                F_bar += w_i * channel.qfim(th_i[0])

        # Apply QCRB-saturating POVM (Theorem V.6 / LAN achievability)
        outcome = channel.sample_outcome(theta_true, rng)

        # Bayesian weight update
        new_w = np.array([
            w_i * channel.likelihood(outcome, th_i)
            for w_i, th_i in zip(weights, particles)
        ])
        w_sum = new_w.sum()

        if w_sum < 1e-300:
            particles = prior.sample(n_particles, rng)
            weights   = np.ones(n_particles) / n_particles
        else:
            weights = new_w / w_sum
            ess = 1.0 / float(np.sum(weights**2))
            if ess < n_particles / 2:
                idx       = rng.choice(n_particles, size=n_particles,
                                       replace=True, p=weights)
                particles = particles[idx].copy()
                weights   = np.ones(n_particles) / n_particles
                particles += rng.normal(0.0, 0.01, size=particles.shape)
                particles[:, 0] = np.clip(particles[:, 0], 1e-4, None)

        mu_k, cov_k = _weighted_stats(particles, weights)
        history_mean.append(mu_k)
        history_cov.append(cov_k)

    theta_hat, cov_final = _weighted_stats(particles, weights)
    return EstimationResult(
        theta_hat    = theta_hat,
        cov_final    = cov_final,
        history_mean = history_mean,
        history_cov  = history_cov,
    )


# =============================================================================
# GRAPHS  (2 separate figures, default matplotlib theme)
# =============================================================================

def plot_mean_convergence(result: EstimationResult,
                          n_bar_true: float, phi_true: float,
                          prior_mean: np.ndarray,
                          out: str) -> None:
    """
    Figure 1 — Posterior mean of n̄ and φ vs measurement round.
    Two subplots side by side; each shows the posterior mean,
    ±1σ band, true value, and prior mean.
    """
    rounds = np.arange(1, len(result.history_mean) + 1)
    means  = np.array([m   for m   in result.history_mean])    # (M, 2)
    stds   = np.array([np.sqrt([c[0,0], c[1,1]])
                       for c in result.history_cov])            # (M, 2)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=False)
    fig.suptitle("Algorithm 1 — Posterior Mean Convergence", fontsize=13)

    specs = [
        (0, n_bar_true, prior_mean[0], "n̄ (mean photon number)", "n̄"),
        (1, phi_true,   prior_mean[1], "φ (phase, rad)",          "φ"),
    ]

    for col, true_val, prior_val, ylabel, param in specs:
        ax = axes[col]
        ax.fill_between(rounds,
                        means[:, col] - stds[:, col],
                        means[:, col] + stds[:, col],
                        alpha=0.20, label="±1σ posterior band")
        ax.plot(rounds, means[:, col], lw=2,       label="Posterior mean")
        ax.axhline(true_val,  ls="--", lw=1.5,     label=f"True {param} = {true_val:.2f}")
        ax.axhline(prior_val, ls=":",  lw=1.2, color="grey",
                   label=f"Prior mean = {prior_val:.2f}")
        ax.set_xlabel("Measurement round k")
        ax.set_ylabel(ylabel)
        ax.set_title(f"Convergence of {param}")
        ax.legend(fontsize=8)
        ax.grid(True, linestyle="--", alpha=0.5)

    fig.tight_layout()
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {out}")


def plot_variance_vs_qcrb(result: EstimationResult,
                          qcrb: np.ndarray,
                          out: str) -> None:
    """
    Figure 2 — Posterior variance of n̄ and φ vs measurement round (log scale).
    Two subplots side by side; each shows the decaying variance and the
    frequentist QCRB = (1/M) F^{-1}(theta_true) as a horizontal reference.
    """
    rounds = np.arange(1, len(result.history_cov) + 1)
    var_nb = np.array([c[0, 0] for c in result.history_cov])
    var_ph = np.array([c[1, 1] for c in result.history_cov])

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=False)
    fig.suptitle("Algorithm 1 — Posterior Variance vs QCRB", fontsize=13)

    specs = [
        (var_nb, qcrb[0, 0], "Var(n̄)  [log scale]", "n̄"),
        (var_ph, qcrb[1, 1], "Var(φ)  [log scale]", "φ"),
    ]

    for ax, (var_data, qcrb_val, ylabel, param) in zip(axes, specs):
        ax.semilogy(rounds, var_data, lw=2, label=f"Posterior Var({param})")
        ax.axhline(qcrb_val, ls="--", lw=1.5,
                   label=f"QCRB = {qcrb_val:.5f}")
        ax.set_xlabel("Measurement round k")
        ax.set_ylabel(ylabel)
        ax.set_title(f"Variance of {param}")
        ax.legend(fontsize=8)
        ax.grid(True, which="both", linestyle="--", alpha=0.5)

    fig.tight_layout()
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {out}")


# =============================================================================
# MAIN
# =============================================================================

def main():
    # Channel and ground-truth parameters
    gamma_loss = 0.02    # ns^-1  (amplitude damping)
    gamma_phi  = 0.005   # ns^-1  (phase damping)
    L          = 10.0    # km
    n_bar_true = 5.0
    phi_true   = 0.30    # rad
    theta_true = np.array([n_bar_true, phi_true])
    M          = 50      # measurement rounds
    n_particles= 300

    channel = FibreChannel(gamma_loss, gamma_phi, L)
    prior   = GaussianPrior(
        mean = np.array([4.0, 0.50]),
        cov  = np.diag([2.0, 0.50]),
    )

    # ---------- Run Algorithm 1 ----------
    print("Running Algorithm 1...")
    result = algorithm1_adaptive_bayesian_estimation(
        channel     = channel,
        prior       = prior,
        M           = M,
        theta_true  = theta_true,
        n_particles = n_particles,
        seed        = 42,
    )

    # ---------- Print results ----------
    qfim = channel.qfim(n_bar_true)
    qcrb = np.linalg.inv(qfim) / M

    print(f"\n  True params : n_bar = {n_bar_true},  phi = {phi_true:.3f} rad")
    print(f"  Prior mean  : n_bar = {prior.mean[0]},  phi = {prior.mean[1]:.3f} rad")
    print(f"  Theta_hat   : n_bar = {result.theta_hat[0]:.4f},  "
          f"phi = {result.theta_hat[1]:.4f} rad")
    print(f"\n  Final posterior covariance:")
    print(f"    Var(n_bar)     = {result.cov_final[0,0]:.6f}")
    print(f"    Var(phi)       = {result.cov_final[1,1]:.6f}")
    print(f"    Cov(n_bar,phi) = {result.cov_final[0,1]:.6f}")
    print(f"\n  Frequentist QCRB (1/M)*F^-1:")
    print(f"    QCRB[n_bar] = {qcrb[0,0]:.6f}")
    print(f"    QCRB[phi  ] = {qcrb[1,1]:.6f}")
    print(f"  (B5) Posterior may be tighter than QCRB — prior contributes information.\n")

    step = max(1, M // 5)
    print(f"  Convergence table (every {step} rounds):")
    print(f"  {'Round':>6}  {'n_bar_hat':>11}  {'phi_hat':>9}  "
          f"{'Var(n_bar)':>12}  {'Var(phi)':>10}")
    for k in range(step - 1, M, step):     # B2 fixed: labels 10,20,30,40,50
        mu  = result.history_mean[k]
        cov = result.history_cov[k]
        print(f"  {k+1:>6}  {mu[0]:>11.4f}  {mu[1]:>9.4f}  "
              f"{cov[0,0]:>12.6f}  {cov[1,1]:>10.6f}")

    # ---------- Save figures ----------
    print("\nSaving figures...")
    plot_mean_convergence(
        result, n_bar_true, phi_true, prior.mean,
        "fig1_posterior_mean_convergence.png",
    )
    plot_variance_vs_qcrb(
        result, qcrb,
        "fig2_posterior_variance_vs_qcrb.png",
    )


if __name__ == "__main__":
    main()

Running Algorithm 1...

  True params : n_bar = 5.0,  phi = 0.300 rad
  Prior mean  : n_bar = 4.0,  phi = 0.500 rad
  Theta_hat   : n_bar = 4.9906,  phi = 0.3106 rad

  Final posterior covariance:
    Var(n_bar)     = 0.037731
    Var(phi)       = 0.002004
    Cov(n_bar,phi) = 0.003214

  Frequentist QCRB (1/M)*F^-1:
    QCRB[n_bar] = 0.081062
    QCRB[phi  ] = 0.001350
  (B5) Posterior may be tighter than QCRB — prior contributes information.

  Convergence table (every 10 rounds):
   Round    n_bar_hat    phi_hat    Var(n_bar)    Var(phi)
      10       4.2216     0.2565      0.285025    0.005472
      20       4.7266     0.2927      0.159329    0.003536
      30       5.0185     0.3189      0.050986    0.003063
      40       5.0547     0.2874      0.050251    0.002382
      50       4.9906     0.3106      0.037731    0.002004

Saving figures...
  Saved → fig1_posterior_mean_convergence.png
  Saved → fig2_posterior_variance_vs_qcrb.png
